# 00 · Create Environment
### Demo · Lay Out the Data: from small files to liquid clustering

This notebook prepares everything the demo needs. The demo runs it with `%run`, so you never open it during the session.

| What | Where | Used in |
|---|---|---|
| Catalog, schema and volume | `lab2401.layout`, volume `files` | everything |
| Three empty folders for the small path tables | `/Volumes/lab2401/layout/files/{zordered, partitioned, clustered}` | Part A (steps 1–8) |
| Two **large managed tables**, same rows, different layout | `orders_unordered`, `orders_clustered` | Part B (step 9, Query Profile) |
| Helpers | `file_stats`, `show_files`, `last_commits` | both parts |

**Run it once before the session.** Building the large tables takes a few minutes; on later runs they are reused (widget `rebuild_large = no`), so `%run` from the demo only resets the small folders and takes seconds.

## 0. Configuration

**Names are literal on purpose.** The demo's SQL cells use these same paths and table names, written out in full, so the audience reads real SQL and not variables.
One catalog per lab, as in the other labs: `lab2401` = topic 24, lab 1. To use another catalog or schema, Find & Replace (Ctrl+H) `lab2401/layout` and `lab2401.layout` **in both notebooks**.

| Widget | Values | Meaning |
|---|---|---|
| `scale` | `small` · `medium` · `large` | Size of the two tables of Part B (Part A is always 1 M rows) |
| `rebuild_large` | `no` · `yes` | `yes` drops and rebuilds the large tables (e.g. after changing `scale`) |

| Scale | Rows | Files in `orders_unordered` | `targetFileSize` of `orders_clustered` | Use |
|---|---|---|---|---|
| `small` | 20 M | 32 | 8 MB | smoke test: pruning visible, timings not meaningful |
| `medium` | 200 M | 128 | 64 MB | **default for the session** |
| `large` | 600 M | 256 | 128 MB | only if `medium` shows no visible time difference in the rehearsal |

In [ ]:
# ── Libraries ────────────────────────────────────────────────────────────────
from pyspark.sql import functions as F
import time

# ── Widgets (Databricks) with plain defaults anywhere else ───────────────────
DEFAULTS = {"scale": "medium", "rebuild_large": "no"}
try:
    dbutils.widgets.dropdown("scale", DEFAULTS["scale"], ["small", "medium", "large"], "Scale (Part B)")
    dbutils.widgets.dropdown("rebuild_large", DEFAULTS["rebuild_large"], ["no", "yes"], "Rebuild large tables")
    params = {k: dbutils.widgets.get(k) for k in DEFAULTS}
except NameError:
    params = dict(DEFAULTS)

# ── Names (literal, the same as in the demo's SQL cells) ─────────────────────
CATALOG     = "lab2401"
SCHEMA      = "layout"
VOLUME_NAME = "files"
VOLUME      = "/Volumes/lab2401/layout/files"

# Part A · small path tables (folders inside the volume, so we can look at their files)
ZORDERED    = "/Volumes/lab2401/layout/files/zordered"
PARTITIONED = "/Volumes/lab2401/layout/files/partitioned"
CLUSTERED   = "/Volumes/lab2401/layout/files/clustered"

# Part B · large Unity Catalog managed tables
ORDERS_UNORDERED = "lab2401.layout.orders_unordered"
ORDERS_CLUSTERED = "lab2401.layout.orders_clustered"

SCALES = {
    "small":  dict(rows=20_000_000,  files=32,  target="8mb"),
    "medium": dict(rows=200_000_000, files=128, target="64mb"),
    "large":  dict(rows=600_000_000, files=256, target="128mb"),
}
SCALE = params["scale"]
CFG = SCALES[SCALE]
REBUILD_LARGE = params["rebuild_large"] == "yes"
print(f"scale={SCALE} ({CFG['rows']:,} rows) | rebuild_large={REBUILD_LARGE}")

## 1. Catalog, schema and volume

The schema gets **Predictive Optimization disabled**. Otherwise the platform could compact or cluster `orders_unordered` on its own between the setup and the demo, and the "before" table would no longer be "before" (see the appendix: Predictive Optimization only acts on managed tables, which is exactly what Part B uses).

In [ ]:
try:
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
except Exception as e:   # no CREATE CATALOG privilege: fine if the catalog already exists
    print(f"CREATE CATALOG skipped ({type(e).__name__}); using the existing catalog {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.`{VOLUME_NAME}`")

try:
    spark.sql(f"ALTER SCHEMA {CATALOG}.{SCHEMA} DISABLE PREDICTIVE OPTIMIZATION")
    PO_STATE = "disabled on the schema"
except Exception as e:
    PO_STATE = (f"NOT disabled ({type(e).__name__}). Before the demo, check that "
                f"{ORDERS_UNORDERED} still has {CFG['files']} files (section 3).")
print("Predictive Optimization:", PO_STATE)

## 2. Part A: start from empty folders

The demo creates the three small tables itself (that is the lesson), so here we only delete what a previous run left behind. Every run of the demo starts from the same state.

In [ ]:
for path in (ZORDERED, PARTITIONED, CLUSTERED):
    dbutils.fs.rm(path, True)
print("Empty folders ready:", ZORDERED, PARTITIONED, CLUSTERED, sep="\n  ")

## 3. Part B: two large tables, same rows, different layout

| Table | How it is written | What every file looks like |
|---|---|---|
| `orders_unordered` | One write, `files` tasks, **optimized writes and auto compaction off** | Each file holds orders from **all** dates (2023-01-01 → 2024-12-31): min/max cannot skip anything on `order_date` |
| `orders_clustered` | Same rows, generated again by the same function, into a table with `CLUSTER BY (order_date)`, then `OPTIMIZE` | Each file holds a **narrow** date range: a one-month filter can skip most files |

The columns are deliberately simple, so the Query Profile is about layout and nothing else:
`order_id` · `customer_id` · `order_date` · `country` · `amount`.

The values come from a hash of `order_id` with a fixed seed, so every rebuild produces exactly the same rows.

**Why generate twice instead of `CREATE TABLE … AS SELECT * FROM orders_unordered`?** A copy would read the whole of `orders_unordered`, and those files would stay in the **disk cache** of the executors. The demo would then compare a warm table A with a cold table B, and the times would lie. Generated separately, the setup never reads either table: both start cold. (Pruning is not affected either way: it is decided from the log before anything is read.)
`order_id` is sequential inside each file (that is how `range` splits); it plays no role in the demo.

In [ ]:
SEED = 42
COUNTRIES = ["NL", "BE", "DE", "FR", "ES", "IT", "PL", "SE"]

def u(col, salt):
    """Deterministic uniform value in [0, 1) from a column, the seed and a salt."""
    h = F.xxhash64(F.col(col), F.lit(SEED), F.lit(salt))
    return F.pmod(h, F.lit(1_000_000_007)) / F.lit(1_000_000_007.0)

def table_exists(name):
    return spark.catalog.tableExists(name)

COLUMNS = "order_id BIGINT, customer_id BIGINT, order_date DATE, country STRING, amount DOUBLE"

def orders_df():
    """The rows of Part B. Deterministic: every call produces exactly the same rows."""
    n, files = CFG["rows"], CFG["files"]
    return (spark.range(0, n, 1, files)                       # one input partition per output file
            .withColumnRenamed("id", "order_id")
            .withColumn("customer_id", (u("order_id", 1) * (n // 20)).cast("long"))
            .withColumn("order_date", F.date_add(F.lit("2023-01-01").cast("date"),
                                                 (u("order_id", 2) * 731).cast("int")))
            .withColumn("country", F.element_at(F.array(*[F.lit(c) for c in COUNTRIES]),
                                                (F.floor(u("order_id", 3) * len(COUNTRIES)) + 1).cast("int")))
            .withColumn("amount", F.round(F.lit(5) + u("order_id", 4) * 495, 2)))

def build_orders_unordered():
    spark.sql(f"DROP TABLE IF EXISTS {ORDERS_UNORDERED}")
    spark.sql(f"""
        CREATE TABLE {ORDERS_UNORDERED} ({COLUMNS})
        TBLPROPERTIES ('delta.autoOptimize.optimizeWrite' = 'false',
                       'delta.autoOptimize.autoCompact'   = 'false')""")
    orders_df().write.mode("append").saveAsTable(ORDERS_UNORDERED)

def build_orders_clustered():
    # Generated again, NOT copied from orders_unordered: the setup never reads A,
    # so neither table starts the demo warm in the disk cache.
    spark.sql(f"DROP TABLE IF EXISTS {ORDERS_CLUSTERED}")
    spark.sql(f"""
        CREATE TABLE {ORDERS_CLUSTERED} ({COLUMNS})
        CLUSTER BY (order_date)
        TBLPROPERTIES ('delta.targetFileSize' = '{CFG['target']}',
                       'delta.autoOptimize.autoCompact' = 'false')""")
    orders_df().write.mode("append").saveAsTable(ORDERS_CLUSTERED)
    spark.sql(f"OPTIMIZE {ORDERS_CLUSTERED}")

t0 = time.time()
if REBUILD_LARGE or not table_exists(ORDERS_UNORDERED):
    print(f"Building {ORDERS_UNORDERED} ({CFG['rows']:,} rows, {CFG['files']} files) ...")
    build_orders_unordered()
else:
    print(f"Reusing {ORDERS_UNORDERED}")
if REBUILD_LARGE or not table_exists(ORDERS_CLUSTERED):
    print(f"Building {ORDERS_CLUSTERED} (CLUSTER BY order_date + OPTIMIZE) ...")
    build_orders_clustered()
else:
    print(f"Reusing {ORDERS_CLUSTERED}")
print(f"Part B ready in {time.time() - t0:,.0f} s")

## 4. Helpers used by the demo

Three small functions, so that the demo shows the evidence without repeating the same query again and again.
The first time each idea appears, the demo writes the SQL out in full; after that it calls the helper.

| Helper | Shows | Equivalent SQL |
|---|---|---|
| `file_stats(t)` | files, total size and **average file size** | `DESCRIBE DETAIL t` → `sizeInBytes / numFiles` |
| `show_files(t, col)` | one row per Parquet file: rows, `min(col)`, `max(col)` | `SELECT _metadata.file_name, count(*), min(col), max(col) FROM t GROUP BY 1` |
| `last_commits(t, n)` | the last `n` commits: operation, files added and removed | `DESCRIBE HISTORY t` → `operationMetrics` |

`t` is any SQL table reference: ``delta.`/Volumes/...` `` for a path table, `catalog.schema.table` for a managed one.
`show_files` computes min/max **from the data** (it reads the column); the `_delta_log` holds the same min/max as statistics, and those are what data skipping uses.

In [ ]:
def path_table(path):
    """SQL reference to a path table: delta.`/Volumes/...`"""
    return f"delta.`{path}`"

def file_stats(t):
    d = spark.sql(f"DESCRIBE DETAIL {t}").collect()[0]
    n, size = d["numFiles"], d["sizeInBytes"]
    avg = size / n if n else 0
    unit = (f"{avg/1024:,.1f} KB" if avg < 1024**2 else f"{avg/1024**2:,.1f} MB")
    print(f"{n:,} files · {size/1024**2:,.1f} MB in total · average file {unit}")

def show_files(t, col):
    display(spark.sql(f"""
        SELECT _metadata.file_name AS file,
               count(*)   AS num_rows,
               min({col}) AS {col}_min,
               max({col}) AS {col}_max
        FROM {t}
        GROUP BY 1
        ORDER BY {col}_min, {col}_max"""))

def last_commits(t, n=1):
    m = F.col("operationMetrics")
    display(spark.sql(f"DESCRIBE HISTORY {t} LIMIT {n}")
            .select("version", "operation",
                    m["numFiles"].alias("files_written"),
                    m["numRemovedFiles"].alias("files_removed"),
                    m["numAddedFiles"].alias("files_added"),
                    "operationParameters"))

## 5. Check before moving on

In [ ]:
for name in (ORDERS_UNORDERED, ORDERS_CLUSTERED):
    print(f"{name:<48}", end=" ")
    file_stats(name)
print()
print(f"Expected: orders_unordered = {CFG['files']} files. Fewer means something compacted it: "
      f"set rebuild_large = yes and run again.")
print("Predictive Optimization:", PO_STATE)

## Cleanup (optional)

Uncomment when you no longer need the demo. Do **not** run it as part of *Run all*.

In [ ]:
# spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.{SCHEMA} CASCADE")